In [ ]:
%reload_ext autoreload
%autoreload 2

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

import os
import logging
import yaml
import random
import time
import ipynbname

import pandas as pd

import ferdinand.utils as utl   
import ferdinand.sqlite_utils as sqlutl
import ferdinand.download_utils as dlutl

from pathlib import Path
from types import SimpleNamespace


# Purpose and Documentation

## Purpose 
Download current image files from the IMPC image server based on the metadata CSV generated by _010_fetch_data_from_impc.ipynb_.

## Prerequisites

- A valid runtime configuration file (_../config/config.yml_) with correctly defined _centers_, _run_name_, _parameters_, and _directories_
- Successful execution of the template setup cells from _NNN_notebook_template.ipynb_
- Existing _../data_ and _../reports/_ directories 
- Existing SQLite database _../data/&lt;run_name&gt;/&lt;all_data_dir&gt;/&lt;sqlite_db&gt;.db_
- Network access to the IMPC image servers (https://www.mousephenotype.org)

## Workflow

1. Clean up ambiguous OMERO ID records
2. Download DICOM and JPEG images
3. Update _&lt;sqlite_db&gt;.db_ with the current image download status

# Definition / Settings

This section initializes the notebook environment: 
- loads the runtime configuration from _../config/config.yml_
- creates the required directory structure
- sets up logging and the SQLite connection

**Do not proceed past this section** </br>
Missing paths and an uninitialized database connection will cause subsequent cells to crash.

## General settings

In [ ]:
# load the configuration file
config_path = Path('../config/config.yml')
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

print(pd.DataFrame(config['settings'].items(), columns=['configuration parameter', 'set to']).to_markdown(tablefmt="pretty", index=False))

## Set up runtime paths

In [ ]:
# create directories as per configuration
dirs = SimpleNamespace(**config['data_directories'])
for key, value in vars(dirs).items():
    dirs.__dict__[key] = os.path.join(config['project_root'], config['data_dir'], config['run_name'], value)
    os.makedirs(dirs.__dict__[key], exist_ok=True)
    if os.path.exists(dirs.__dict__[key]):
        print(f" Directory created: {dirs.__dict__[key]}")
    else:
        print(f" ERROR: Directory does not exist: {dirs.__dict__[key]}")

## Store runtime configuration when necessary

In [ ]:
# copy the current '../config/config.yml' to '../reports/'
dest_filename = f"config_{config['run_name']}.yml"
dest_dir = os.path.join(config['project_root'], config['reports_dir'], config['run_name'])
os.makedirs(dest_dir, exist_ok=True)
dest_path = os.path.join(dest_dir, dest_filename)
if not os.path.exists(dest_path): 
    commmand_string = f"cp {config_path} {dest_path}"
    os.system(commmand_string)

    if os.path.exists(dest_path): 
        print(f" Current runtime configuration \'{config_path}' saved to \'{dest_path}\'")
    else:
        print(f" ERROR: could not save configuration file to \'{dest_path}\'")

## Set up logger

In [ ]:
# setup logging
notebook_name = ipynbname.name()
notebook_idx = notebook_name.split('_')[0]

log_path = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], 'logs')
os.makedirs(log_path, exist_ok=True)

log_file = os.path.join(log_path, f"{notebook_name}.log")
print(f"Log file: {log_file}")

try:
    logger = utl.setup_logger(log_file, log_level=logging.INFO)
    print("Logger successfully set up.")
except Exception as e:
    print(f"Error setting up logger: {e}")

## Set up SQLite connection

In [ ]:
# expected SQLlite database file
sqlite_file = os.path.join(dirs.all_data_dir, f"{config['sqlite_db']}.db")

# if already exists, connect to it
if os.path.exists(sqlite_file):
    conn = sqlutl.connect_db(sqlite_file)
    if conn is not None:
        print(f"connected to database {sqlite_file}\n")
else: 
    print(f" ERROR: No SQLite database found!")

# Clean up entries with ambiguous OMERO identifiers

In [ ]:
# remove database records that have ambiguous OMERO IDs (e.g. same ID linked to multiple mouse IDs)
sqlutl.remove_files_with_ambiguous_omero_ids(conn, db_table='impc_data', logger=logger)

# Download image files and update database records

In [ ]:
# download settings
# mode: REAL=download everything, TEST=only 20 rows
mode = config["run_mode"]
print(f"mode is {mode}")

## Get current download status

In [ ]:
total_number = len(sqlutl.select_rows(conn, db_table='impc_data'))
downloaded_df = sqlutl.select_rows_by_column(conn, db_table='impc_data', column='downloaded', value='yes', logger=logger)
downloaded_number = len(downloaded_df)
not_yet_downloaded_number = len(sqlutl.select_rows_by_column(conn, db_table='impc_data', column='downloaded', value='no', logger=logger))

print(f"             total number of images: {total_number}")
print(f"        number of downloaded images: {downloaded_number}")
print(f"number of not yet downloaded images: {not_yet_downloaded_number}")

## Check whether downloaded images exist

In [ ]:
if downloaded_number > 0:
    try: 
        for idx, row in downloaded_df.iterrows():
            local_img_path = utl.build_local_image_dir(dirs.images_dir, row.center, row['cohort_type'], row['gene_symbol'], row['sex'])
            local_img_file = os.path.join(local_img_path, f"{row['omero_id']}.{row['file_extension']}")
            if not os.path.exists(local_img_file): 
                print(f" missing file for omero ID {row['omero_id']} at center {row.center}")
    except NameError as e:
        print(f":-( Error occurred: {e}")

## Fetch the next download batch

In [ ]:
to_download = sqlutl.select_next_download_batch(conn, db_table='impc_data', limit=200000)

print(f" Number of files to download:\n  {len(to_download)}")
print(f" Available file types:\n  {to_download.impc_file_type.unique()}")

In [ ]:
logger.setLevel(logging.DEBUG)
print("preparing download ...")

if mode == 'TEST':
    print(f" Number of files to download:\n {len(to_download)}")

for idx, row in to_download.iterrows():

    # random wait time - be nice to the image server
    wait_time = random.uniform(0.1, 2)
    time.sleep(wait_time)
    
    # handle genotype
    gene = f"{row['gene_symbol'].replace(' ', '_')}/" if row['cohort_type']=='mutant' else ''

    # build center/genotype/gene/sex-specific path ..
    local_image_dir = utl.build_local_image_dir(dirs.images_dir, 
                                               row['center'].replace(' ', '_'),
                                               row['cohort_type'], 
                                               gene, 
                                               row['sex'])            
    
    # ... and make sure path exists
    os.makedirs(local_image_dir, exist_ok=True)

    try: 
        print(f"Downloading file {idx} with omero id {row['omero_id']}")
        _df = dlutl.download_image(row['omero_id'], local_image_dir, False, True, logger)
        _jpeg_df = dlutl.download_image(row['omero_id'], local_image_dir, True, True, logger)
        info_df = pd.merge(left=_df, right=_jpeg_df, on='omero_id', how='outer')
        sqlutl.save_full_image_metadata(info_df, conn, 'impc_data') 
        sqlutl.update_column_values(conn, db_table='impc_data', column='downloaded', value='yes', condition_column='omero_id', condition_value=row['omero_id'])
    except RuntimeError: 
        sqlutl.update_column_values(conn, db_table='impc_data', column='downloaded', value='failed', condition_column='omero_id', condition_value=row['omero_id'])
        print(f"ERROR: Failed to download {local_image_dir}!")

In [ ]:
# check that all downloaded files are present
files = utl.select_jpegs_by_column(conn, 
                                   db_table='impc_data', 
                                   column='downloaded', 
                                   value='yes', 
                                   source_path=dirs.images_dir)
for file in files:
    if not os.path.exists(file):
        print(f" missing file {file}")

# Update CSV file

In [ ]:
# update the CSV file with the new download information
updated_df = pd.read_sql_query("SELECT * from impc_data", conn)

data_file = os.path.join(dirs.all_data_dir, f'{config["sqlite_db"]}.csv')
updated_df.to_csv(data_file, index=False)
print(f" CSV file {data_file} updated")

# Create thumbnail gallery

In [ ]:
# select all downloaded files for further processing
rows = sqlutl.select_rows_by_column(conn, 'impc_data', 'downloaded', 'yes')

In [ ]:
# create thumbnails for all downloaded images
utl.create_and_save_thumbnails(
    data=rows,
    centers=config['centers'],
    path2images=dirs.images_dir,
    path2thumbnails=dirs.thumbnails_dir
)

In [ ]:
# create an HTML gallery directory
gallery_dir = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], 'thumbnails_gallery')
os.makedirs(gallery_dir, exist_ok=True)

In [ ]:
# create HTML gallery of downloaded images
utl.create_thumbnail_gallery(
    data=rows,
    html_file_name=f'{notebook_idx}_all_downloaded_images.html',
    title='Overview of Downloaded Images',
    html_dir=gallery_dir,
    thumbnails_dir=dirs.thumbnails_dir,
    source_dir=dirs.images_dir)

In [ ]:
# create HTML galleries of downloaded images for each center
for center in config['centers']:
    center_df = rows[rows['center'] == center]
    if not center_df.empty:
        center_stripped = center.replace(" ", "_")
        center_gallery_dir = os.path.join(gallery_dir, center_stripped)
        os.makedirs(center_gallery_dir, exist_ok=True)
        utl.create_thumbnail_gallery(
            data=center_df,
            html_file_name=f'{notebook_idx}_all_downloaded_images_{center_stripped}.html',
            title=f'Overview of Downloaded Images from {center}',
            html_dir=center_gallery_dir,
            thumbnails_dir=dirs.thumbnails_dir,
            source_dir=dirs.images_dir)